# Phase 3 · Submission S4 — Architecture D (Dilated Residual TCN) + Prompt Tuning

**Task:** Rhetorical Role prediction — one of 13 labels for every sentence of a legal judgment (CL and IT domains).
**Primary metric:** Macro-F1.

**Method**
1. **Backbone:** the Phase-1 Architecture D temporal convolutional network (10 residual blocks of two causal, weight-normalised, dilated convolutions; dilations 1…512; 1024 channels; 114.2M parameters), loaded from `checkpoints/best.pt`. All Phase-1 weights stay frozen.
2. **Adaptation — Prompt Tuning (Lester et al., 2021):** 20 trainable "soft prompt" vectors are placed in front of every sentence's token embeddings. Because the convolutions are causal with a receptive field of ~4,000 tokens, every token of the sentence sees the prompt, so the prompt steers how the frozen TCN encodes it. The prompt is initialised from the embeddings of real vocabulary tokens. Only the prompt (20 × 1024 = 20,480 numbers) is trained in the backbone. Prompt tuning needs no change inside the network, which is why it suits a convolutional model where attention-based adapters (LoRA on q/v, prefix tuning) do not apply.
3. **Sentence vector:** the TCN states of the sentence's own tokens (prompt positions removed); mean ⊕ final `[EOS]` state, projected to 512 dimensions. Padding is on the right, so causal convolutions never let `[PAD]` affect real tokens.
4. **Document context and training:** identical to S1–S3 (document BiLSTM, CL/IT domain embedding, relative position, class-weighted cross-entropy, windows of 128 sentences).

**Metric reporting:** dev Macro-F1 over all 13 labels (`macro_f1`, selects the best epoch) and over present labels only (`macro_f1_present`).

**Before running (Kaggle)**
- Accelerator: **GPU T4** (one GPU is used).
- Add-ons → Secrets: **`HF`** = your Hugging Face token.
- Add Input → datasets: **`ruchikhemka/data-archd`** (Architecture D) and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- To resume an interrupted run: add the previous version's output of this notebook as an input; Cell 11 picks up `checkpoints/latest.pt` automatically.

Outputs are written to `/kaggle/working/P3_S4_D_TCN_PromptTuning/` — `submission.zip` is the file to upload to Codabench.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P3_S4_D_TCN_PromptTuning"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Sentence encoding: 128 tokens cover ~98% of sentences (see data analysis).
MAX_SENT_TOKENS = 128
# Training windows of consecutive sentences, and sentences per forward pass at inference.
WINDOW_SENTS = 128
WINDOW_STRIDE = 64
ENCODE_CHUNK = 256

# Prompt tuning: number of soft prompt vectors placed before every sentence.
PROMPT_LEN = 20
PROMPT_INIT_VOCAB = 5000     # prompt initialised from embeddings of random tokens among the 5,000 most frequent ids

# Classification head (same as S1).
SENT_DIM = 512
DOC_LSTM_HIDDEN = 256
DROPOUT = 0.2

# Optimisation (same as S1–S3 except the adapter learning rate: soft prompts need a large step).
EPOCHS = 20
LR_ADAPTER = 1e-2
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
PATIENCE = 5                 # stop after this many epochs without a better dev Macro-F1
CLASS_WEIGHT_POWER = 0.5     # weight_c = (N / (K * count_c)) ** power, capped below
CLASS_WEIGHT_MAX = 10.0
USE_AMP = DEVICE.type == "cuda"

print("Run:", RUN_NAME)
print("Outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download the Phase 3 data from Hugging Face using the Kaggle secret "HF".
# Only the Phase_3_RR folder is fetched; files already present are not downloaded again.
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=HF_TOKEN,
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_3_RR/**"])

P3_DIR = DATA_DIR / "Phase_3_RR"
REQUIRED_FILES = ["Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json",
                  "Train/IT_dev.json", "Train/label_vocab.json",
                  "Test/CL_test.json", "Test/IT_test.json"]
missing = [f for f in REQUIRED_FILES if not (P3_DIR / f).is_file()]
assert not missing, f"Missing Phase 3 files: {missing}"
print("Phase 3 data ready in", P3_DIR)

In [ ]:
# Cell 4 — Load train/dev/test documents and the official label vocabulary.
# Each document becomes {id, domain, sents, labels}; test documents have labels = None.
LABEL2ID = json.load(open(P3_DIR / "Train/label_vocab.json"))
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
DOMAINS = {"CL": 0, "IT": 1}

def load_labelled(path, domain):
    docs = []
    for d in json.load(open(path)):
        assert len(d["text"]) == len(d["labels"]), f"Length mismatch in {d['id']}"
        docs.append({"id": d["id"], "domain": domain, "sents": d["text"],
                     "labels": [LABEL2ID[label] for label in d["labels"]]})
    return docs

def load_test(path, domain):
    return [{"id": doc_id, "domain": domain, "sents": sents, "labels": None}
            for doc_id, sents in json.load(open(path)).items()]

train_docs = load_labelled(P3_DIR / "Train/CL_train.json", "CL") + load_labelled(P3_DIR / "Train/IT_train.json", "IT")
dev_docs = load_labelled(P3_DIR / "Train/CL_dev.json", "CL") + load_labelled(P3_DIR / "Train/IT_dev.json", "IT")
test_docs = load_test(P3_DIR / "Test/CL_test.json", "CL") + load_test(P3_DIR / "Test/IT_test.json", "IT")

for name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    print(f"{name:5s}: {len(docs):3d} documents, {sum(len(d['sents']) for d in docs):6d} sentences")

In [ ]:
# Cell 5 — Load the shared Phase-1 BPE tokenizer and turn every sentence into token ids.
# Each sentence becomes [BOS] + at most 126 BPE tokens + [EOS]; [PAD] (id 0) is used for padding.
tokenizer_paths = sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))
assert tokenizer_paths, "Add the dataset rohitbatra2004/cs781-phase1-tokenized-corpus as an input."
tokenizer = Tokenizer.from_file(tokenizer_paths[0])

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 2, 3), "Unexpected special-token ids"
assert tokenizer.get_vocab_size() == 50000, "Unexpected vocabulary size"

def encode_sentences(sents):
    encodings = tokenizer.encode_batch(sents)
    return [[BOS_ID] + e.ids[:MAX_SENT_TOKENS - 2] + [EOS_ID] for e in encodings]

for d in train_docs + dev_docs + test_docs:
    d["ids"] = encode_sentences(d["sents"])

lengths = [len(x) for d in train_docs for x in d["ids"]]
print("Tokenizer:", tokenizer_paths[0])
print(f"Train sentence length: mean {np.mean(lengths):.1f} tokens; "
      f"{100 * np.mean([n == MAX_SENT_TOKENS for n in lengths]):.1f}% truncated at {MAX_SENT_TOKENS}")

In [ ]:
# Cell 6 — Rebuild the Phase-1 Architecture D TCN and strictly load its pretrained weights.
# Weights and configuration both come from checkpoints/best.pt (keys "model_state_dict" and "model_config").
d_paths = sorted(glob.glob("/kaggle/input/**/architecture_D/checkpoints/best.pt", recursive=True))
assert d_paths, "Add the dataset ruchikhemka/data-archd as an input."
D_CKPT = torch.load(d_paths[0], map_location="cpu", mmap=True, weights_only=False)
D_CONFIG = D_CKPT["model_config"]
assert D_CONFIG["embedding_dim"] == D_CONFIG["channels"] and D_CONFIG["convs_per_block"] == 2

class WeightNormConvParams(nn.Module):
    # Holds the Phase-1 weight-normalised convolution exactly as saved: weight = g * v / ||v||.
    def __init__(self, channels, kernel_size):
        super().__init__()
        self.weight_g = nn.Parameter(torch.ones(channels, 1, 1))
        self.weight_v = nn.Parameter(0.02 * torch.randn(channels, channels, kernel_size))  # overwritten by Phase-1 weights
        self.bias = nn.Parameter(torch.zeros(channels))

    def weight(self):
        return self.weight_g * self.weight_v / self.weight_v.norm(dim=(1, 2), keepdim=True)

class CausalConv1d(nn.Module):
    # Dilated causal convolution computed as one matrix multiply per kernel tap, on [batch, time, channels].
    # Mathematically identical to Phase 1's left-padded Conv1d, but a tap whose shift is >= the sequence length
    # only ever reads padding zeros and is skipped. Sentences are <= 148 positions while dilations reach 512,
    # so this avoids the very slow cuDNN path for large dilations on short inputs.
    def __init__(self, channels, kernel_size, dilation):
        super().__init__()
        self.kernel_size, self.dilation = kernel_size, dilation
        self.conv = WeightNormConvParams(channels, kernel_size)

    def forward(self, x):
        T = x.size(1)
        w = self.conv.weight()
        out = self.conv.bias
        for j in range(self.kernel_size):
            shift = (self.kernel_size - 1 - j) * self.dilation
            if shift >= T:
                continue
            shifted = F.pad(x, (0, 0, shift, 0))[:, :T] if shift > 0 else x
            out = out + shifted @ w[:, :, j].t()
        return out

class TCNResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv1d(channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(channels, kernel_size, dilation)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        out = self.dropout1(F.relu(self.conv1(x)))
        out = self.dropout2(F.relu(self.conv2(out)))
        return F.relu(out + x)

class ArchitectureDBackbone(nn.Module):
    # Phase-1 TCN language model without the vocabulary projection.
    # forward(input_ids) returns the state of every sentence token: [batch, tokens, channels].
    # soft_prompt stays None until prompt tuning is attached in Cell 7.
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.tcn_blocks = nn.ModuleList([TCNResidualBlock(cfg["channels"], cfg["kernel_size"], d, cfg["dropout"])
                                         for d in cfg["dilations"]])
        self.soft_prompt = None

    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        n_prompt = 0
        if self.soft_prompt is not None:
            n_prompt = self.soft_prompt.size(0)
            x = torch.cat([self.soft_prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], dim=1)
        for block in self.tcn_blocks:
            x = block(x)
        return x[:, n_prompt:]

backbone = ArchitectureDBackbone(D_CONFIG)
backbone.load_state_dict(D_CKPT["model_state_dict"], strict=True)
del D_CKPT

n_backbone = sum(p.numel() for p in backbone.parameters())
assert n_backbone == int(D_CONFIG["estimated_parameter_count"]), "Parameter count differs from Phase-1 config"
receptive_field = 1 + 2 * (D_CONFIG["kernel_size"] - 1) * sum(D_CONFIG["dilations"])
print(json.dumps({k: v for k, v in D_CONFIG.items() if not isinstance(v, (dict, list)) or k == "dilations"}, indent=2))
print(f"Loaded Phase-1 Architecture D from {Path(d_paths[0]).parent} — {n_backbone:,} parameters, "
      f"receptive field {receptive_field} tokens")

In [ ]:
# Cell 7 — Attach prompt tuning: PROMPT_LEN trainable vectors prepended to every sentence.
# Initialised from embeddings of real (frequent) tokens, which is more stable than random vectors.
# Freeze all Phase-1 weights first; only the soft prompt created below will train.
for p in backbone.parameters():
    p.requires_grad_(False)

init_ids = torch.randint(4, PROMPT_INIT_VOCAB, (PROMPT_LEN,), generator=torch.Generator().manual_seed(SEED))
backbone.soft_prompt = nn.Parameter(backbone.token_embedding.weight[init_ids].detach().clone())

backbone = backbone.to(DEVICE)

# Check on the actual device that the prompt changes the output and receives a gradient.
probe_ids = torch.randint(4, 1000, (2, 16), device=DEVICE)
out = backbone(probe_ids)
assert out.shape == (2, 16, D_CONFIG["channels"]), "Prompt positions were not removed from the output"
out.float().pow(2).mean().backward()
assert backbone.soft_prompt.grad is not None and backbone.soft_prompt.grad.abs().sum() > 0, \
    "Soft prompt receives no gradient"
backbone.zero_grad(set_to_none=True)
del out

n_prompt = backbone.soft_prompt.numel()
print(f"Soft prompt attached: {PROMPT_LEN} vectors × {D_CONFIG['embedding_dim']} = {n_prompt:,} trainable "
      f"backbone parameters ({100 * n_prompt / n_backbone:.3f}% of the backbone)")

In [ ]:
# Cell 8 — Hierarchical rhetorical-role model on top of the prompted TCN (same head as S1–S3).
# Sentence vector = projection of [mean state ; [EOS] state]; a BiLSTM over the document gives context.
class RhetoricalRoleModel(nn.Module):
    def __init__(self, backbone, hidden_size):
        super().__init__()
        self.backbone = backbone
        self.sent_proj = nn.Sequential(nn.Linear(2 * hidden_size, SENT_DIM), nn.GELU(), nn.Dropout(DROPOUT))
        self.domain_emb = nn.Embedding(len(DOMAINS), SENT_DIM)
        self.doc_lstm = nn.LSTM(SENT_DIM + 1, DOC_LSTM_HIDDEN, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(2 * DOC_LSTM_HIDDEN, NUM_LABELS)

    def encode_sentences(self, ids, mask):
        """ids, mask: [S, T] -> sentence vectors [S, SENT_DIM]."""
        states = self.backbone(ids)
        m = mask.unsqueeze(-1).to(states.dtype)
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1.0)
        last = states[torch.arange(states.size(0), device=states.device), mask.sum(1) - 1]
        return self.sent_proj(torch.cat([mean, last], dim=-1))

    def classify(self, sent_vecs, positions, domain_id):
        """sent_vecs: [S, SENT_DIM] of one document (or window) -> logits [S, NUM_LABELS]."""
        x = sent_vecs + self.domain_emb.weight[domain_id]
        x = torch.cat([x, positions.unsqueeze(-1).to(x.dtype)], dim=-1).unsqueeze(0)
        context, _ = self.doc_lstm(x)
        return self.classifier(self.dropout(context.squeeze(0)))

model = RhetoricalRoleModel(backbone, D_CONFIG["channels"]).to(DEVICE)
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%)")

In [ ]:
# Cell 9 — Helpers: padding, relative positions, training windows and class weights.
# Windows of 128 consecutive sentences (stride 64) let a document fit in one training step.
def pad_batch(id_lists):
    width = max(len(x) for x in id_lists)
    ids = torch.full((len(id_lists), width), PAD_ID, dtype=torch.long)
    for i, x in enumerate(id_lists):
        ids[i, :len(x)] = torch.tensor(x)
    return ids, ids.ne(PAD_ID)

def rel_positions(n):
    return torch.arange(n, dtype=torch.float) / max(n - 1, 1)

def make_windows(docs):
    windows = []
    for doc_index, d in enumerate(docs):
        n = len(d["ids"])
        starts = list(range(0, max(n - WINDOW_SENTS, 0) + 1, WINDOW_STRIDE))
        if starts[-1] + WINDOW_SENTS < n:
            starts.append(n - WINDOW_SENTS)
        windows += [(doc_index, s, min(s + WINDOW_SENTS, n)) for s in starts]
    return windows

train_windows = make_windows(train_docs)

label_counts = Counter(label for d in train_docs for label in d["labels"])
n_train_sents = sum(label_counts.values())
class_weights = torch.ones(NUM_LABELS)
for label_id in range(NUM_LABELS):
    if label_counts[label_id] > 0:
        w = (n_train_sents / (NUM_LABELS * label_counts[label_id])) ** CLASS_WEIGHT_POWER
        class_weights[label_id] = min(w, CLASS_WEIGHT_MAX)
CLASS_WEIGHTS = class_weights.to(DEVICE)

print(f"Training windows per epoch: {len(train_windows)}")
for label_id in range(NUM_LABELS):
    print(f"  {ID2LABEL[label_id]:24s} count={label_counts[label_id]:5d}  weight={class_weights[label_id]:.2f}")

In [ ]:
# Cell 10 — Prediction and evaluation over whole documents.
# macro_f1 averages over all 13 labels (matches Codabench for S1); macro_f1_present over labels that occur.
ALL_LABELS = list(range(NUM_LABELS))

@torch.no_grad()
def predict_docs(model, docs):
    model.eval()
    all_preds = []
    for d in docs:
        vectors = []
        for start in range(0, len(d["ids"]), ENCODE_CHUNK):
            ids, mask = pad_batch(d["ids"][start:start + ENCODE_CHUNK])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                vectors.append(model.encode_sentences(ids.to(DEVICE), mask.to(DEVICE)).float())
        vectors = torch.cat(vectors)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model.classify(vectors, rel_positions(len(d["ids"])).to(DEVICE), DOMAINS[d["domain"]])
        all_preds.append(logits.argmax(-1).tolist())
    return all_preds

def macro_scores(gold, pred):
    return {"macro_f1": f1_score(gold, pred, labels=ALL_LABELS, average="macro", zero_division=0),
            "macro_p": precision_score(gold, pred, labels=ALL_LABELS, average="macro", zero_division=0),
            "macro_r": recall_score(gold, pred, labels=ALL_LABELS, average="macro", zero_division=0),
            "macro_f1_present": f1_score(gold, pred, average="macro", zero_division=0)}

def evaluate(model, docs):
    preds = predict_docs(model, docs)
    result = {}
    for domain in ("CL", "IT", "ALL"):
        gold = [y for d in docs if domain in (d["domain"], "ALL") for y in d["labels"]]
        pred = [y for d, p in zip(docs, preds) if domain in (d["domain"], "ALL") for y in p]
        scores = macro_scores(gold, pred)
        if domain == "ALL":
            result.update(scores)
        else:
            result[f"{domain}_macro_f1_present"] = scores["macro_f1_present"]
    return result, preds

print("Evaluation helpers ready (13-label Macro-F1 for model selection; present-label Macro-F1 also reported).")

In [ ]:
# Cell 11 — Optimiser, learning-rate schedule and checkpoint resume.
# latest.pt is saved every epoch; if a previous run's output is attached as input, training resumes from it.
adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "soft_prompt" in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "soft_prompt" not in n]
optimizer = torch.optim.AdamW([{"params": adapter_params, "lr": LR_ADAPTER},
                               {"params": head_params, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)

total_steps = EPOCHS * len(train_windows)
warmup_steps = int(WARMUP_FRACTION * total_steps)
def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / max(warmup_steps, 1)
    return max(0.0, (total_steps - step) / max(total_steps - warmup_steps, 1))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def trainable_state(model):
    full = model.state_dict()
    return {name: full[name].detach().cpu() for name in TRAINABLE_NAMES}

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best.pt"

# Copy checkpoints from an attached earlier run of this notebook, if there is one.
if not LATEST_CKPT.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], LATEST_CKPT)
        if Path(previous[0]).with_name("best.pt").exists():
            shutil.copy(Path(previous[0]).with_name("best.pt"), BEST_CKPT)
        print("Copied checkpoints from", Path(previous[0]).parent)

train_state = {"epoch": 0, "best_f1": -1.0, "best_epoch": 0, "bad_epochs": 0,
               "history": [], "train_seconds": 0.0}
if LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    train_state = ckpt["train_state"]
    print(f"Resumed after epoch {train_state['epoch']} (best dev Macro-F1 {train_state['best_f1']:.4f})")
else:
    print("No checkpoint found — training starts from the Phase-1 weights.")
print(f"Total optimisation steps: {total_steps} (warm-up {warmup_steps})")

In [ ]:
# Cell 12 — Training loop: one window per step, class-weighted cross-entropy, mixed precision.
# After every epoch: evaluate dev Macro-F1 (13 labels), keep best.pt, save latest.pt, stop early after PATIENCE bad epochs.
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

for epoch in range(train_state["epoch"], EPOCHS):
    if train_state["bad_epochs"] >= PATIENCE:
        print(f"Early stopping: no dev improvement for {PATIENCE} epochs.")
        break
    model.train()
    order = train_windows[:]
    random.Random(SEED + epoch).shuffle(order)
    start_time, loss_sum = time.time(), 0.0

    for doc_index, s, e in order:
        d = train_docs[doc_index]
        ids, mask = pad_batch(d["ids"][s:e])
        labels = torch.tensor(d["labels"][s:e], device=DEVICE)
        positions = rel_positions(len(d["ids"]))[s:e].to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            vectors = model.encode_sentences(ids.to(DEVICE), mask.to(DEVICE))
            logits = model.classify(vectors, positions, DOMAINS[d["domain"]])
        loss = F.cross_entropy(logits.float(), labels, weight=CLASS_WEIGHTS)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(adapter_params + head_params, GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_sum += loss.item()

    dev_scores, _ = evaluate(model, dev_docs)
    epoch_seconds = time.time() - start_time
    train_state["train_seconds"] += epoch_seconds
    train_state["epoch"] = epoch + 1
    record = {"epoch": epoch + 1, "train_loss": loss_sum / len(order), **dev_scores, "seconds": epoch_seconds}
    train_state["history"].append(record)

    if dev_scores["macro_f1"] > train_state["best_f1"]:
        train_state.update(best_f1=dev_scores["macro_f1"], best_epoch=epoch + 1, bad_epochs=0)
        torch.save({"trainable": trainable_state(model), "epoch": epoch + 1, "dev": dev_scores}, BEST_CKPT)
    else:
        train_state["bad_epochs"] += 1

    torch.save({"trainable": trainable_state(model), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "train_state": train_state}, LATEST_CKPT)
    print(f"Epoch {epoch + 1:2d} | loss {record['train_loss']:.4f} | dev Macro-F1 {dev_scores['macro_f1']:.4f} "
          f"(present-labels {dev_scores['macro_f1_present']:.4f}; CL {dev_scores['CL_macro_f1_present']:.4f}, "
          f"IT {dev_scores['IT_macro_f1_present']:.4f}) | {epoch_seconds / 60:.1f} min"
          + ("  <- best" if train_state["best_epoch"] == epoch + 1 else ""))

print(f"Best dev Macro-F1 (13 labels) {train_state['best_f1']:.4f} at epoch {train_state['best_epoch']}")

In [ ]:
# Cell 13 — Reload the best epoch and report detailed dev results (per class and confusion matrix).
# These numbers go into the report's Phase 3 analysis; they are also saved to outputs/.
best = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
dev_scores, dev_preds = evaluate(model, dev_docs)

gold_flat = [y for d in dev_docs for y in d["labels"]]
pred_flat = [y for p in dev_preds for y in p]
report = classification_report(gold_flat, pred_flat, labels=ALL_LABELS,
                               target_names=[ID2LABEL[i] for i in ALL_LABELS], zero_division=0)
print(f"Best epoch {best['epoch']} — dev Macro-F1 {dev_scores['macro_f1']:.4f} (13 labels) | "
      f"P {dev_scores['macro_p']:.4f} | R {dev_scores['macro_r']:.4f} | "
      f"present-labels Macro-F1 {dev_scores['macro_f1_present']:.4f}")
print(report)

cm = confusion_matrix(gold_flat, pred_flat, labels=ALL_LABELS)
json.dump({"dev_scores": dev_scores, "best_epoch": best["epoch"], "history": train_state["history"],
           "confusion_matrix": cm.tolist(), "labels": [ID2LABEL[i] for i in ALL_LABELS]},
          open(OUT_DIR / "dev_metrics.json", "w"), indent=2)
json.dump({d["id"]: [ID2LABEL[y] for y in p] for d, p in zip(dev_docs, dev_preds)},
          open(OUT_DIR / "dev_predictions.json", "w"), indent=2)
open(OUT_DIR / "dev_classification_report.txt", "w").write(report)
print("Saved dev metrics, predictions and report to", OUT_DIR)

In [ ]:
# Cell 14 — Ablation for the report: the same sentence vectors WITHOUT document context.
# A linear classifier on frozen best-model sentence vectors; this does not change the submission.
@torch.no_grad()
def sentence_vectors(docs):
    model.eval()
    out = []
    for d in docs:
        for start in range(0, len(d["ids"]), ENCODE_CHUNK):
            ids, mask = pad_batch(d["ids"][start:start + ENCODE_CHUNK])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out.append(model.encode_sentences(ids.to(DEVICE), mask.to(DEVICE)).float())
    return torch.cat(out)

X_train, y_train = sentence_vectors(train_docs), torch.tensor([y for d in train_docs for y in d["labels"]], device=DEVICE)
X_dev, y_dev = sentence_vectors(dev_docs), [y for d in dev_docs for y in d["labels"]]

linear_probe = nn.Linear(SENT_DIM, NUM_LABELS).to(DEVICE)
probe_opt = torch.optim.AdamW(linear_probe.parameters(), lr=1e-2, weight_decay=WEIGHT_DECAY)
for _ in range(200):
    probe_opt.zero_grad()
    F.cross_entropy(linear_probe(X_train), y_train, weight=CLASS_WEIGHTS).backward()
    probe_opt.step()
sentence_only = macro_scores(y_dev, linear_probe(X_dev).argmax(-1).tolist())

print(f"Dev Macro-F1 (13 labels)  sentence-only: {sentence_only['macro_f1']:.4f}   "
      f"with document BiLSTM: {dev_scores['macro_f1']:.4f}")
json.dump({"sentence_only": sentence_only, "with_context": dev_scores},
          open(OUT_DIR / "context_ablation.json", "w"), indent=2)

In [ ]:
# Cell 15 — Predict the 10 test documents, validate the output and build submission.zip.
# prediction.json maps every test document id to one label per sentence, in the original order.
test_preds = predict_docs(model, test_docs)
prediction = {d["id"]: [ID2LABEL[y] for y in p] for d, p in zip(test_docs, test_preds)}

expected_ids = set(json.load(open(P3_DIR / "Test/CL_test.json"))) | set(json.load(open(P3_DIR / "Test/IT_test.json")))
assert set(prediction) == expected_ids, "Test document ids do not match"
for d in test_docs:
    assert len(prediction[d["id"]]) == len(d["sents"]), f"Sentence count mismatch for {d['id']}"
assert all(label in LABEL2ID for labels in prediction.values() for label in labels), "Invalid label found"

PREDICTION_PATH = WORK_DIR / "prediction.json"
SUBMISSION_PATH = WORK_DIR / "submission.zip"
json.dump(prediction, open(PREDICTION_PATH, "w"), indent=2)
with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_PATH, arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_PATH) as zf:
    assert zf.namelist() == ["prediction.json"], "prediction.json must be at the zip root"
test_counts = Counter(label for labels in prediction.values() for label in labels)
print(f"Validated {len(prediction)} documents, {sum(test_counts.values())} sentences -> {SUBMISSION_PATH}")
print("Predicted label distribution:", dict(test_counts.most_common()))

In [ ]:
# Cell 16 — Run summary for the README, the experiment log and results/submissions.csv.
# Records the cost (parameters, time, GPU memory) next to the dev scores.
summary = {
    "run": RUN_NAME, "backbone": "Architecture D — dilated residual TCN (Phase 1)",
    "technique": "Prompt tuning", "prompt_len": PROMPT_LEN,
    "trainable_params": n_trainable, "total_params": n_total, "prompt_params": n_prompt,
    "best_epoch": best["epoch"], "epochs_run": train_state["epoch"],
    "train_minutes": round(train_state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "dev": dev_scores, "dev_sentence_only": sentence_only,
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))